# CodeLens — 1-Click Free Cloud GPU Backend

Run CodeLens's Python backend and open-weight AI model (Gemma 3 4B) with **free T4 GPU acceleration**.

### How to run:
1. In the top menu, go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.
2. Click the **Play (▶)** button on the code cell below (or **Runtime > Run all**).
3. Once ready, click the generated `https://codlenz.vercel.app/?backend=...` link!

In [ ]:
# --- 1-Click Setup: Ollama + GPU + Cloudflare Tunnel + CodeLens ---
import os, re, subprocess, time

print("⏳ [1/5] Installing Ollama and Cloudflare Tunnel...")
!curl -fsSL https://ollama.com/install.sh | sh
!curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

print("⏳ [2/5] Starting Ollama server in background...")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

print("⏳ [3/5] Downloading Gemma 3 4B model (with GPU acceleration)...")
!ollama pull gemma3:4b || ollama pull qwen2.5:3b

print("⏳ [4/5] Installing CodeLens...")
!pip install -q git+https://github.com/manju342r/codlenz.git

print("⏳ [5/5] Creating secure public HTTPS tunnel...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

backend_url = None
for line in iter(tunnel.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        backend_url = match.group(0)
        print("\n" + "=" * 70)
        print(f"🎉 SUCCESS! BACKEND IS LIVE AT: {backend_url}")
        print(f"👉 CLICK TO OPEN VERCEL: https://codlenz.vercel.app/?backend={backend_url}")
        print(f"👉 CLICK TO OPEN HUGGING FACE: https://manju342r-codlenz.hf.space/?backend={backend_url}")
        print("=" * 70 + "\n")
        break

# Start CodeLens demo server with CORS enabled
os.environ["CORS_ORIGIN"] = "*"
!codelens serve --port 8000